In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Semantic Cache — Demo Notebook\n",
    "**VII Semester B.Tech CSE Project | VIT Vellore**  \n",
    "**Team:** Shayla & Lohith\n",
    "\n",
    "This notebook demonstrates the Semantic Cache system end-to-end:\n",
    "1. Query classification with adaptive thresholds\n",
    "2. Cache miss → LLM call → save to cache\n",
    "3. Cache hit → instant return (no LLM call)\n",
    "4. Response time comparison: cache hit vs miss\n",
    "5. Borderline similarity — how adaptive thresholds make smarter decisions than a fixed threshold"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Setup: add the project root to Python path so we can import from src/\n",
    "# Run this notebook from the project root: VII_Sem_Project/\n",
    "# Or from notebooks/ subfolder — both cases are handled below.\n",
    "\n",
    "import sys\n",
    "import os\n",
    "\n",
    "# Walk up until we find the src/ folder (handles running from any subfolder)\n",
    "project_root = os.getcwd()\n",
    "if not os.path.exists(os.path.join(project_root, 'src')):\n",
    "    project_root = os.path.dirname(os.getcwd())\n",
    "\n",
    "if project_root not in sys.path:\n",
    "    sys.path.insert(0, project_root)\n",
    "\n",
    "# Change working directory so ChromaDB finds ./chroma_db at the project root\n",
    "os.chdir(project_root)\n",
    "print(f\"Project root: {project_root}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import our modules\n",
    "import time\n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.patches as mpatches\n",
    "\n",
    "from dotenv import load_dotenv\n",
    "load_dotenv()\n",
    "\n",
    "from src.classifier import classify_query\n",
    "from src.embedder import get_embedding\n",
    "from src.cache import check_cache, save_to_cache\n",
    "\n",
    "print(\"All modules loaded successfully.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 1. Query Classification — Adaptive Thresholds\n",
    "The classifier assigns each query to a type and returns the appropriate similarity threshold."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "test_queries = [\n",
    "    \"What is machine learning?\",\n",
    "    \"Who invented the telephone?\",\n",
    "    \"How many planets are in the solar system?\",\n",
    "    \"How does a neural network learn?\",\n",
    "    \"Why does the sky appear blue?\",\n",
    "    \"Compare supervised and unsupervised learning\",\n",
    "    \"Write a short poem about artificial intelligence\",\n",
    "    \"Create a story about a robot\",\n",
    "    \"Suggest five project ideas for machine learning\",\n",
    "]\n",
    "\n",
    "print(f\"{'Query':<50} {'Category':<12} {'Threshold'}\")\n",
    "print(\"-\" * 75)\n",
    "for q in test_queries:\n",
    "    category, threshold = classify_query(q)\n",
    "    print(f\"{q:<50} {category:<12} {threshold}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 2. Cache Miss → LLM Call → Saved to Cache"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "from groq import Groq\n",
    "\n",
    "groq_client = Groq(api_key=os.getenv(\"GROQ_API_KEY\"))\n",
    "GROQ_MODEL = \"openai/gpt-oss-20b\"\n",
    "\n",
    "def ask(query: str):\n",
    "    \"\"\"Full semantic cache lookup pipeline — mirrors src/main.py logic.\"\"\"\n",
    "    start = time.time()\n",
    "    \n",
    "    category, threshold = classify_query(query)\n",
    "    cached_answer, similarity, _ = check_cache(query)\n",
    "    \n",
    "    if cached_answer:\n",
    "        elapsed = round((time.time() - start) * 1000)\n",
    "        return {\n",
    "            \"answer\": cached_answer,\n",
    "            \"cache_hit\": True,\n",
    "            \"category\": category,\n",
    "            \"threshold\": threshold,\n",
    "            \"similarity_score\": similarity,\n",
    "            \"response_time_ms\": elapsed\n",
    "        }\n",
    "    \n",
    "    # Cache miss — call LLM\n",
    "    response = groq_client.chat.completions.create(\n",
    "        model=GROQ_MODEL,\n",
    "        messages=[\n",
    "            {\"role\": \"system\", \"content\": \"You are a helpful assistant. Answer clearly and concisely.\"},\n",
    "            {\"role\": \"user\", \"content\": query}\n",
    "        ]\n",
    "    )\n",
    "    answer = response.choices[0].message.content\n",
    "    save_to_cache(query, answer)\n",
    "    \n",
    "    elapsed = round((time.time() - start) * 1000)\n",
    "    return {\n",
    "        \"answer\": answer,\n",
    "        \"cache_hit\": False,\n",
    "        \"category\": category,\n",
    "        \"threshold\": threshold,\n",
    "        \"similarity_score\": similarity,\n",
    "        \"response_time_ms\": elapsed\n",
    "    }\n",
    "\n",
    "# First call — cache miss\n",
    "query1 = \"What is deep learning?\"\n",
    "result1 = ask(query1)\n",
    "\n",
    "print(f\"Query       : {query1}\")\n",
    "print(f\"Cache Hit   : {result1['cache_hit']}\")\n",
    "print(f\"Category    : {result1['category']}\")\n",
    "print(f\"Threshold   : {result1['threshold']}\")\n",
    "print(f\"Similarity  : {result1['similarity_score']}\")\n",
    "print(f\"Time (ms)   : {result1['response_time_ms']}\")\n",
    "print(f\"\\nAnswer:\\n{result1['answer']}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 3. Cache Hit — Same Query Asked Again"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Same query — should be a cache hit with similarity ~1.0\n",
    "result2 = ask(query1)\n",
    "\n",
    "print(f\"Query       : {query1}\")\n",
    "print(f\"Cache Hit   : {result2['cache_hit']}\")\n",
    "print(f\"Similarity  : {result2['similarity_score']}\")\n",
    "print(f\"Time (ms)   : {result2['response_time_ms']}\")\n",
    "print(f\"\\nSpeedup     : {result1['response_time_ms'] / max(result2['response_time_ms'], 1):.1f}x faster\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 4. Response Time Comparison — Multiple Queries"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Run a set of queries twice — first time (miss), second time (hit)\n",
    "demo_queries = [\n",
    "    \"What is a neural network?\",\n",
    "    \"Define gradient descent\",\n",
    "    \"What are transformers in NLP?\",\n",
    "]\n",
    "\n",
    "miss_times = []\n",
    "hit_times = []\n",
    "labels = []\n",
    "\n",
    "for q in demo_queries:\n",
    "    r_miss = ask(q)        # First call — cache miss\n",
    "    r_hit  = ask(q)        # Second call — cache hit\n",
    "    miss_times.append(r_miss['response_time_ms'])\n",
    "    hit_times.append(r_hit['response_time_ms'])\n",
    "    labels.append(q[:30] + \"...\" if len(q) > 30 else q)\n",
    "    print(f\"Miss: {r_miss['response_time_ms']:>5} ms  |  Hit: {r_hit['response_time_ms']:>4} ms  |  {q}\")\n",
    "\n",
    "print(f\"\\nAverage miss: {sum(miss_times)/len(miss_times):.0f} ms\")\n",
    "print(f\"Average hit : {sum(hit_times)/len(hit_times):.0f} ms\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Bar chart: Cache Miss vs Cache Hit response times\n",
    "x = range(len(labels))\n",
    "width = 0.35\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(10, 5))\n",
    "bars1 = ax.bar([i - width/2 for i in x], miss_times, width, label='Cache Miss (LLM call)', color='#e74c3c')\n",
    "bars2 = ax.bar([i + width/2 for i in x], hit_times,  width, label='Cache Hit (instant)',   color='#2ecc71')\n",
    "\n",
    "ax.set_xlabel('Query')\n",
    "ax.set_ylabel('Response Time (ms)')\n",
    "ax.set_title('Semantic Cache: Response Time — Cache Miss vs Cache Hit')\n",
    "ax.set_xticks(list(x))\n",
    "ax.set_xticklabels(labels, rotation=15, ha='right')\n",
    "ax.legend()\n",
    "ax.bar_label(bars1, fmt='%d ms', padding=3)\n",
    "ax.bar_label(bars2, fmt='%d ms', padding=3)\n",
    "plt.tight_layout()\n",
    "plt.savefig('notebooks/response_time_comparison.png', dpi=150)\n",
    "plt.show()\n",
    "print(\"Chart saved to notebooks/response_time_comparison.png\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 5. Adaptive Threshold — Why One Fixed Threshold Fails\n",
    "\n",
    "This is the **core novelty** of the project. We send a query similar (but not identical) to a cached one,\n",
    "and show how the category-specific threshold decides correctly whether to use the cache."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save a factual query to cache first\n",
    "seed_query  = \"What is photosynthesis?\"\n",
    "seed_answer = \"Photosynthesis is the process by which plants convert sunlight, water, and CO2 into glucose and oxygen.\"\n",
    "save_to_cache(seed_query, seed_answer)\n",
    "print(f\"Saved to cache: '{seed_query}'\\n\")\n",
    "\n",
    "# Now test semantically similar but differently typed queries\n",
    "similar_queries = [\n",
    "    \"What is photosynthesis?\",                       # identical → factual, similarity ~1.0 → HIT\n",
    "    \"Define photosynthesis\",                          # very similar → factual, similarity ~0.95 → HIT\n",
    "    \"Can you explain photosynthesis simply?\",         # similar but analytical → similarity ~0.80, threshold 0.88 → MISS\n",
    "    \"Write a poem about photosynthesis\",              # creative → similarity ~0.75, threshold 0.80 → MISS\n",
    "]\n",
    "\n",
    "print(f\"{'Query':<45} {'Category':<12} {'Threshold':<10} {'Similarity':<12} {'Decision'}\")\n",
    "print(\"-\" * 95)\n",
    "\n",
    "for q in similar_queries:\n",
    "    category, threshold = classify_query(q)\n",
    "    cached, similarity, _ = check_cache(q)\n",
    "    decision = \"✅ HIT\" if cached else \"❌ MISS\"\n",
    "    print(f\"{q:<45} {category:<12} {threshold:<10} {similarity:<12} {decision}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Interpretation\n",
    "\n",
    "| Scenario | Similarity | Threshold | Decision | Why it's correct |\n",
    "|---|---|---|---|---|\n",
    "| Identical factual query | ~1.00 | 0.95 | HIT ✅ | Exact same question — cached answer is perfect |\n",
    "| `Define photosynthesis` | ~0.95 | 0.95 | HIT ✅ | Same factual meaning, cached answer still valid |\n",
    "| `Explain photosynthesis simply` | ~0.80 | 0.88 | MISS ❌ | Analytical query — needs a specific explanatory style |\n",
    "| `Write a poem about photosynthesis` | ~0.75 | 0.80 | MISS ❌ | Creative query — a factual definition is not a poem |\n",
    "\n",
    "**With a fixed threshold of 0.80:** the `explain` query would incorrectly return a factual definition as the answer.  \n",
    "**Our adaptive system:** recognises it as analytical (threshold 0.88), correctly calls the LLM for a proper explanation."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## Summary\n",
    "\n",
    "| Metric | Value |\n",
    "|---|---|\n",
    "| Embedding model | `all-MiniLM-L6-v2` (384-dim) |\n",
    "| Vector store | ChromaDB (persistent, cosine similarity) |\n",
    "| LLM | Groq `openai/gpt-oss-20b` |\n",
    "| Factual threshold | 0.95 |\n",
    "| Analytical threshold | 0.88 |\n",
    "| Creative threshold | 0.80 |\n",
    "| Avg. cache miss time | ~1000–1500 ms |\n",
    "| Avg. cache hit time | ~10–50 ms |\n",
    "| Speedup on hit | **20–100×** |"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.14.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

: 